### Supplier-Side Pricing Features Impact (GMV + Bookings)

This notebook is designed to be **presentation-ready** for a Product Manager.

#### What it answers
1. **What does feature usage look like?** Adoption rates, intensity, and combinations.
2. **Do suppliers using features have higher bookings and GMV?** (descriptive + stratified comparisons)
3. **Where should Product focus?** Segment and market pockets with high upside (adoption gaps + outcome differences).

#### Guardrails (as requested)
- **Supplier-side only** (GMV and bookings).
- **No repeat customer metrics**.
- **No regressions and no model-based propensity scoring**.
- Uses **only**:
  - `production.supply_analytics.pricing_feature_audit_base`
  - `production.supply_analytics.pricing_feature_audit_performance`

#### Interpretation
- Comparisons are **associations**, not causal effects.
- To reduce obvious selection bias **without regressions**, we add **stratified comparisons** (within segment/activity/location buckets) and show the distribution of within-bucket lifts.

In [0]:

# ----------------------------
# Notebook setup (plot defaults)
# ----------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cycler

# Enable grid and update its appearance
plt.rcParams.update({'axes.grid': True})
plt.rcParams.update({'grid.color': 'silver'})
plt.rcParams.update({'grid.linestyle': '--'})

# Set figure resolution
plt.rcParams.update({'figure.dpi': 150})

# Hide the top and right spines
plt.rcParams.update({'axes.spines.top': False})
plt.rcParams.update({'axes.spines.right': False})

# Increase font sizes
plt.rcParams.update({'font.size': 12})  # General font size
plt.rcParams.update({'axes.titlesize': 14})  # Title font size
plt.rcParams.update({'axes.labelsize': 12})  # Axis label font size

plt.rcParams.update({'axes.prop_cycle': cycler.cycler('color', ['#FF5533'])})

plt.style.use('default')
plt.rcParams.update({
    'figure.dpi': 140,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Simple, consistent colors
PALETTE = {
    'primary': '#2E86AB',
    'secondary': '#A23B72',
    'accent': '#F18F01',
    'good': '#06A77D',
    'bad': '#D62828',
    'neutral': '#6C757D',
}

def _maybe_wrap(labels, width=18):
    """Hard-wrap long labels to avoid overlap."""
    import textwrap
    return ['\n'.join(textwrap.wrap(str(x), width=width)) for x in labels]

def _fmt_eur(x):
    if pd.isna(x):
        return 'NA'
    x = float(x)
    if abs(x) >= 1_000_000:
        return f"€{x/1_000_000:.1f}M"
    if abs(x) >= 1_000:
        return f"€{x/1_000:.1f}k"
    return f"€{x:,.0f}"

def _winsor(s, p=0.01):
    """Winsorize series for stable plots."""
    lo = s.quantile(p)
    hi = s.quantile(1-p)
    return s.clip(lo, hi)

### 1) Load base tables and create analysis datasets

We join configuration to performance at `tour_id`.

**Primary population**: tours on **native pricing** (`uses_external_pricing == 0`).

In [0]:

# Load base tables
config_df = spark.sql("""
    SELECT *
    FROM production.supply_analytics.pricing_feature_audit_base
""").toPandas()

perf_df = spark.sql("""
    SELECT *
    FROM production.supply_analytics.pricing_feature_audit_performance
""").toPandas()

# Keep only supplier-side performance outcomes (no repeat customer fields)
perf_keep = ['tour_id','gmv_l12mo','bookings_l12mo','tickets_l12mo','avg_booking_value_l12mo','nr_l12mo']
perf_df = perf_df[[c for c in perf_keep if c in perf_df.columns]].copy()

# Merge
raw = config_df.merge(perf_df, on='tour_id', how='left')

# Coerce numerics we will use
num_cols = [
    'gmv_l12mo','bookings_l12mo','tickets_l12mo','avg_booking_value_l12mo','nr_l12mo',
    'num_individual_categories','num_group_categories','num_addons','total_price_tiers',
    'pricing_dates_covered_next_12mo','pct_dates_with_vacancy_info','cutoff_hours','sample_max_vacancies'
]
for c in num_cols:
    if c in raw.columns:
        raw[c] = pd.to_numeric(raw[c], errors='coerce')

# Normalize feature flags
feature_flags = [
    'has_individual_pricing','has_group_pricing','has_addons_configured','has_any_scale_pricing',
    'has_capacity_limits','has_cutoff_configured',
    'has_min_participant_requirement','has_max_participant_limit',
    'uses_prices_over_api','uses_live_availability_and_price','uses_pricing_scales_over_api',
    'uses_external_pricing'
]
for f in feature_flags:
    if f in raw.columns:
        raw[f] = raw[f].fillna(0).astype(int)

# Native pricing slice
if 'uses_external_pricing' in raw.columns:
    df = raw[raw['uses_external_pricing'] == 0].copy()
else:
    df = raw.copy()

print('Rows (native pricing slice):', len(df))
print('Unique tours:', df['tour_id'].nunique())
print('Unique suppliers:', df['supplier_id'].nunique())

# Tour-level dataset
seg_fields = [c for c in ['supplier_id','segment','activity_type','tour_category','location_name'] if c in df.columns]

agg = {f:'max' for f in feature_flags if f in df.columns}
for c in seg_fields:
    agg[c] = 'first'

for c in ['num_individual_categories','num_group_categories','num_addons','total_price_tiers',
          'pricing_dates_covered_next_12mo','pct_dates_with_vacancy_info','cutoff_hours','sample_max_vacancies']:
    if c in df.columns:
        agg[c] = 'mean'

outcomes = [c for c in ['gmv_l12mo','bookings_l12mo','tickets_l12mo','avg_booking_value_l12mo','nr_l12mo'] if c in df.columns]
for c in outcomes:
    agg[c] = 'first'

tour = df.groupby('tour_id').agg(agg).reset_index()
for c in outcomes:
    tour[c] = tour[c].fillna(0)

# Performance slice used for impact comparisons (editable)
tour_perf = tour[tour['gmv_l12mo'] > 0].copy()

print('Tour-level rows:', len(tour))
print('Tours w/ GMV>0:', len(tour_perf))

### 1.1 Quick marketplace snapshot (visual)

In [0]:

# Simple KPI bar chart (visual)
labels = ['Tours (native)','Suppliers (native)','Tours w/ GMV>0']
values = [tour['tour_id'].nunique(), tour['supplier_id'].nunique(), tour_perf['tour_id'].nunique()]

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(labels, values, color=PALETTE['primary'], edgecolor='black')
ax.set_title('Analysis population size')
ax.set_ylabel('Count')
ax.set_xticklabels(_maybe_wrap(labels, 14))
for i, v in enumerate(values):
    ax.text(i, v, f"  {v:,}", va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

### 2) Define core feature set + usage intensity

We focus on a consistent, PM-friendly set:
- Individual pricing, Group pricing, Addons, Scale pricing, Capacity limits, Cutoff time

We also keep API/live flags available for segmentation.

In [0]:

label_map = {
    'has_individual_pricing': 'Individual pricing',
    'has_group_pricing': 'Group pricing',
    'has_addons_configured': 'Addons',
    'has_any_scale_pricing': 'Scale pricing',
    'has_capacity_limits': 'Capacity limits',
    'has_cutoff_configured': 'Cutoff time',
    'uses_prices_over_api': 'Prices over API',
    'uses_live_availability_and_price': 'Live availability & price',
    'uses_pricing_scales_over_api': 'Scales over API',
}

core_features = [
    'has_individual_pricing','has_group_pricing','has_addons_configured',
    'has_any_scale_pricing','has_capacity_limits','has_cutoff_configured'
]
core_features = [c for c in core_features if c in tour.columns]

# Core feature count
if core_features:
    tour['core_feature_count'] = tour[core_features].sum(axis=1)
    tour_perf['core_feature_count'] = tour_perf[core_features].sum(axis=1)
else:
    tour['core_feature_count'] = 0
    tour_perf['core_feature_count'] = 0

# Pricing model label
if {'has_individual_pricing','has_group_pricing'}.issubset(tour.columns):
    tour['pricing_model'] = np.select(
        [
            (tour['has_individual_pricing']==1) & (tour['has_group_pricing']==1),
            (tour['has_individual_pricing']==1) & (tour['has_group_pricing']==0),
            (tour['has_individual_pricing']==0) & (tour['has_group_pricing']==1),
        ],
        ['Flexible (Individual + Group)','Individual Only','Group Only'],
        default='None'
    )
    tour_perf['pricing_model'] = np.select(
        [
            (tour_perf['has_individual_pricing']==1) & (tour_perf['has_group_pricing']==1),
            (tour_perf['has_individual_pricing']==1) & (tour_perf['has_group_pricing']==0),
            (tour_perf['has_individual_pricing']==0) & (tour_perf['has_group_pricing']==1),
        ],
        ['Flexible (Individual + Group)','Individual Only','Group Only'],
        default='None'
    )
else:
    tour['pricing_model'] = 'Unknown'
    tour_perf['pricing_model'] = 'Unknown'

# Bundle string (PM-friendly)

def bundle_label(r):
    base = 'Flexible' if (int(r.get('has_individual_pricing',0)) and int(r.get('has_group_pricing',0))) else (
        'Individual' if int(r.get('has_individual_pricing',0)) else (
            'Group' if int(r.get('has_group_pricing',0)) else 'None'
        )
    )

    enh = []
    if int(r.get('has_addons_configured',0)):
        enh.append('Addons')
    if int(r.get('has_any_scale_pricing',0)):
        enh.append('Scale')
    if int(r.get('has_capacity_limits',0)):
        enh.append('Capacity')
    if int(r.get('has_cutoff_configured',0)):
        enh.append('Cutoff')

    if base == 'None' and not enh:
        return 'No pricing features'
    return base if not enh else base + ' + ' + '+'.join(enh)

bundle_cols = set(core_features)
if bundle_cols:
    tour['bundle'] = tour.apply(bundle_label, axis=1)
    tour_perf['bundle'] = tour_perf.apply(bundle_label, axis=1)
else:
    tour['bundle'] = 'Unknown'
    tour_perf['bundle'] = 'Unknown'

tour['pricing_model'].value_counts(dropna=False)

### 3) Feature adoption (tour-level) + how adoption varies by supplier size

A PM typically wants:
- which features are already widely used
- which features have low adoption (and therefore adoption upside)
- whether adoption is concentrated among larger suppliers

In [0]:

# Tour-level adoption bar
feat_cols = [c for c in label_map.keys() if c in tour.columns]

adopt = pd.DataFrame({
    'feature': feat_cols,
    'adoption_rate_pct': [tour[c].mean()*100 for c in feat_cols],
    'tours_using': [int((tour[c]==1).sum()) for c in feat_cols],
    'tours_total': [len(tour)]*len(feat_cols)
})
adopt['feature_name'] = adopt['feature'].map(label_map)

adopt = adopt.sort_values('adoption_rate_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 0.55*len(adopt)+2))
ax.barh(adopt['feature_name'], adopt['adoption_rate_pct'], color=PALETTE['primary'], edgecolor='black')
ax.set_title('Feature adoption (tour-level)')
ax.set_xlabel('Adoption rate (% of native-pricing tours)')
ax.set_xlim(0, 100)

for i, (pct, n) in enumerate(zip(adopt['adoption_rate_pct'], adopt['tours_using'])):
    ax.text(pct + 1, i, f"{pct:.1f}%  ({n:,} tours)", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [0]:

# Supplier-level dataset (needed for supplier-size adoption cuts)
base = tour_perf.copy()  # use GMV>0 tours for stable outcomes; switch to `tour` if needed

supplier_agg = {
    'tour_id': 'count',
    'gmv_l12mo': 'sum',
    'bookings_l12mo': 'sum',
}

for c in feat_cols:
    supplier_agg[c] = 'mean'  # share of supplier tours using the feature

supplier = base.groupby('supplier_id').agg(supplier_agg).reset_index()

supplier = supplier.rename(columns={
    'tour_id': 'tours_active',
    'gmv_l12mo': 'gmv_l12mo_total',
    'bookings_l12mo': 'bookings_l12mo_total'
})

supplier['gmv_per_tour'] = supplier['gmv_l12mo_total'] / supplier['tours_active']
supplier['bookings_per_tour'] = supplier['bookings_l12mo_total'] / supplier['tours_active']

# Supplier intensity: avg number of core features across tours
if core_features:
    supplier['core_feature_intensity'] = supplier[core_features].sum(axis=1)
else:
    supplier['core_feature_intensity'] = 0

supplier['tours_active'].describe()

In [0]:

# Supplier-level adoption: % of suppliers using feature on >=1 tour
sup_adopt = []
for c in feat_cols:
    sup_adopt.append({
        'feature': c,
        'feature_name': label_map.get(c, c),
        'suppliers_using_pct': (supplier[c] > 0).mean()*100,
        'suppliers_using': int((supplier[c] > 0).sum()),
        'suppliers_total': len(supplier)
    })

sup_adopt = pd.DataFrame(sup_adopt).sort_values('suppliers_using_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 0.55*len(sup_adopt)+2))
ax.barh(sup_adopt['feature_name'], sup_adopt['suppliers_using_pct'], color=PALETTE['secondary'], edgecolor='black')
ax.set_title('Feature adoption (supplier-level: used on >=1 tour)')
ax.set_xlabel('% of suppliers')
ax.set_xlim(0, 100)

for i, (pct, n) in enumerate(zip(sup_adopt['suppliers_using_pct'], sup_adopt['suppliers_using'])):
    ax.text(pct + 1, i, f"{pct:.1f}%  ({n:,} suppliers)", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

### 3.1 Usage intensity distribution (suppliers)

This shows whether feature usage is a long tail (most suppliers use few features) or broad-based.

In [0]:

fig, ax = plt.subplots(figsize=(10,5))
ax.hist(supplier['core_feature_intensity'].fillna(0), bins=np.arange(0, 8, 0.5), edgecolor='black', color=PALETTE['accent'])
ax.set_title('Supplier core-feature intensity distribution')
ax.set_xlabel('Average # of core features across supplier tours')
ax.set_ylabel('Suppliers')
plt.tight_layout()
plt.show()

In [0]:

# Adoption vs supplier size (tours_active) using binned scatter

supplier2 = supplier.copy()

# Create size bins (quantiles) to avoid noisy single-point scatter
supplier2['size_bin'] = pd.qcut(supplier2['tours_active'], q=10, duplicates='drop')

bin_summary = supplier2.groupby('size_bin', observed=True).agg(
    suppliers=('supplier_id','count'),
    tours_active_mean=('tours_active','mean'),
    intensity_mean=('core_feature_intensity','mean'),
    gmv_per_tour_mean=('gmv_per_tour','mean'),
    bookings_per_tour_mean=('bookings_per_tour','mean'),
).reset_index()

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(bin_summary['tours_active_mean'], bin_summary['intensity_mean'], marker='o', linewidth=2, color=PALETTE['primary'])
ax.set_title('Supplier size vs pricing-feature intensity (binned)')
ax.set_xlabel('Avg # tours per supplier (within size bin)')
ax.set_ylabel('Avg core-feature intensity')
for x, y, n in zip(bin_summary['tours_active_mean'], bin_summary['intensity_mean'], bin_summary['suppliers']):
    ax.text(x, y, f"  n={n}", va='center', fontsize=8)
plt.tight_layout()
plt.show()

### 4) Do suppliers get more bookings and GMV with feature usage?

We provide three views:

1. **Raw with vs without** (tour-level): easy to read, but can be biased.
2. **Stratified lift distributions** (within segment/activity/location buckets): reduces obvious bias without regressions.
3. **Supplier-level outcome gradients** (scatter and binned charts): does higher intensity correlate with higher outcomes?

We focus on:
- GMV per tour and bookings per tour (more comparable)
- plus total GMV/bookings at supplier level (scale)

In [0]:

# ----------------------------
# 4.1 Raw with-vs-without lifts (tour-level)
# ----------------------------

def bootstrap_lift(df_in, feature, outcome, n_boot=400, trim_p=0.01, seed=7):
    """Bootstrap mean lift (%) using winsorized outcomes for stability."""
    rng = np.random.default_rng(seed)
    a = df_in[df_in[feature]==1][outcome].dropna()
    b = df_in[df_in[feature]==0][outcome].dropna()
    if len(a) < 200 or len(b) < 200:
        return None

    a = _winsor(a, trim_p).values
    b = _winsor(b, trim_p).values

    lifts = []
    for _ in range(n_boot):
        sa = rng.choice(a, size=len(a), replace=True).mean()
        sb = rng.choice(b, size=len(b), replace=True).mean()
        lifts.append(((sa/sb)-1)*100 if sb>0 else np.nan)

    lifts = np.array(lifts)
    lifts = lifts[~np.isnan(lifts)]
    return {
        'lift_mean_pct': float(np.mean(lifts)),
        'ci_low_pct': float(np.quantile(lifts, 0.025)),
        'ci_high_pct': float(np.quantile(lifts, 0.975)),
        'n_with': int(len(a)),
        'n_without': int(len(b)),
    }

impact_rows = []
for f in core_features:
    for outcome in ['gmv_l12mo','bookings_l12mo']:
        if f in tour_perf.columns and outcome in tour_perf.columns:
            res = bootstrap_lift(tour_perf, f, outcome)
            if res:
                impact_rows.append({
                    'feature': f,
                    'feature_name': label_map.get(f,f),
                    'outcome': outcome,
                    **res
                })

impact = pd.DataFrame(impact_rows)
impact

In [0]:

# Plot: raw tour-level lifts with 95% bootstrap CI
for outcome in ['gmv_l12mo','bookings_l12mo']:
    d = impact[impact['outcome']==outcome].copy().sort_values('lift_mean_pct')
    if len(d)==0:
        continue

    y = np.arange(len(d))
    x = d['lift_mean_pct'].values
    xerr = np.vstack([x - d['ci_low_pct'].values, d['ci_high_pct'].values - x])

    fig, ax = plt.subplots(figsize=(10, 0.6*len(d)+2))
    ax.errorbar(x, y, xerr=xerr, fmt='o', color=PALETTE['primary'], ecolor='black', elinewidth=1.5, capsize=4)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(d['feature_name'])
    ax.set_title(f'Raw association: lift in {outcome} (with vs without)')
    ax.set_xlabel('Lift (%) with 95% bootstrap CI (winsorized means)')

    for i, (m, n1) in enumerate(zip(d['lift_mean_pct'], d['n_with'])):
        ax.text(m, i, f"  {m:.1f}% (with: {n1:,})", va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

### 4.2 Stratified lift distributions (no regressions)

We compute **within-bucket** lifts and show their distribution.

Buckets used (if available):
- `segment`
- `activity_type`
- `location_name` (top locations only; long tail can be too sparse)

This answers a PM question directly: are lifts consistent across contexts, or only coming from a few pockets?

In [0]:

# Build stratification columns (choose what exists)
strata_cols = [c for c in ['segment','activity_type'] if c in tour_perf.columns]

# For location, include only top locations by tour count to avoid noisy long tail
if 'location_name' in tour_perf.columns:
    top_locs = tour_perf['location_name'].value_counts().head(25).index
    strat_loc = tour_perf['location_name'].where(tour_perf['location_name'].isin(top_locs), other='Other/Long tail')
    tour_perf['_loc_top'] = strat_loc
    strata_cols_loc = strata_cols + ['_loc_top']
else:
    strata_cols_loc = strata_cols

print('Strata used:', strata_cols_loc)


def stratum_lifts(df_in, feature, outcome, strata, min_per_group=40):
    """Compute per-stratum lift (mean) for with vs without feature."""
    d = df_in[[feature, outcome] + strata].copy()
    d = d.dropna(subset=[outcome])

    g = d.groupby(strata + [feature]).agg(
        n=('tour_id','count') if 'tour_id' in df_in.columns else ('gmv_l12mo','count'),
        mean=(outcome,'mean'),
        median=(outcome,'median')
    ).reset_index()

    # Pivot to with/without
    pv = g.pivot_table(index=strata, columns=feature, values=['n','mean','median'], aggfunc='first')

    # Require both sides and minimum counts
    if (('n',0) not in pv.columns) or (('n',1) not in pv.columns):
        return None

    pv = pv.dropna(subset=[('n',0),('n',1),('mean',0),('mean',1)])
    pv = pv[(pv[('n',0)] >= min_per_group) & (pv[('n',1)] >= min_per_group)].copy()
    if len(pv)==0:
        return None

    pv['lift_mean_pct'] = (pv[('mean',1)] / pv[('mean',0)] - 1) * 100
    pv['lift_median_pct'] = (pv[('median',1)] / pv[('median',0)] - 1) * 100

    # Weight by stratum size
    pv['weight'] = (pv[('n',0)] + pv[('n',1)]).astype(float)
    pv = pv.reset_index()

    keep = strata + ['lift_mean_pct','lift_median_pct','weight',('n',0),('n',1)]
    pv = pv[keep]
    pv.columns = [str(c) for c in pv.columns]
    return pv

# Compute and visualize for each core feature
for feature in core_features:
    for outcome in ['gmv_l12mo','bookings_l12mo']:
        lifts = stratum_lifts(tour_perf, feature, outcome, strata_cols_loc)
        if lifts is None or len(lifts) < 10:
            continue

        # Clip extreme lifts for plotting stability
        v = lifts['lift_mean_pct'].replace([np.inf,-np.inf], np.nan).dropna()
        v_clip = v.clip(v.quantile(0.02), v.quantile(0.98))

        fig, ax = plt.subplots(figsize=(10,4))
        ax.hist(v_clip, bins=25, edgecolor='black', color=PALETTE['neutral'])
        ax.axvline(v.mean(), color=PALETTE['primary'], linewidth=2, label=f"Mean lift: {v.mean():.1f}%")
        ax.axvline(np.median(v), color=PALETTE['accent'], linewidth=2, label=f"Median lift: {np.median(v):.1f}%")
        ax.axvline(0, color='black', linewidth=1)
        ax.set_title(f"Stratified lifts distribution | {label_map.get(feature,feature)} | outcome={outcome}
(strata: {', '.join(strata_cols_loc)})")
        ax.set_xlabel('Within-stratum lift (%)')
        ax.set_ylabel('Number of strata')
        ax.legend()
        plt.tight_layout()
        plt.show()

### 4.3 Supplier-level gradients (scatter + binned lines)

We visualize whether suppliers with higher feature intensity tend to have higher:
- GMV per tour
- bookings per tour

We use:
- a scatter with log scales (to reduce skew)
- a binned line chart to show the average trend

In [0]:

# Scatter: intensity vs GMV per tour (log scale)
# (Sample for rendering speed if needed)
plot_df = supplier.copy()
if len(plot_df) > 15000:
    plot_df = plot_df.sample(15000, random_state=7)

x = plot_df['core_feature_intensity'].fillna(0)
y = plot_df['gmv_per_tour'].replace([np.inf,-np.inf], np.nan).fillna(0)

fig, ax = plt.subplots(figsize=(10,6))
ax.scatter(x, y, s=18, alpha=0.25, color=PALETTE['primary'], edgecolor='none')
ax.set_title('Supplier intensity vs GMV per tour')
ax.set_xlabel('Avg # of core features across supplier tours')
ax.set_ylabel('GMV per tour (L12M)')
ax.set_yscale('log')
ax.set_ylim(1, max(10, np.nanpercentile(y, 99)))
plt.tight_layout()
plt.show()

In [0]:

# Binned trend: intensity vs GMV per tour and bookings per tour

s = supplier.copy()
# Discrete bins on intensity
s['int_bin'] = pd.cut(s['core_feature_intensity'].fillna(0), bins=np.arange(-0.01, 7.01, 0.5))
trend = s.groupby('int_bin', observed=True).agg(
    suppliers=('supplier_id','count'),
    gmv_per_tour=('gmv_per_tour','median'),
    bookings_per_tour=('bookings_per_tour','median'),
).reset_index()

trend = trend[trend['suppliers'] >= 30].copy()
trend['int_mid'] = trend['int_bin'].apply(lambda x: x.mid)

# GMV trend
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(trend['int_mid'], trend['gmv_per_tour'], marker='o', linewidth=2, color=PALETTE['primary'])
ax.set_title('Median GMV per tour vs supplier feature intensity (binned)')
ax.set_xlabel('Avg # core features (bin mid)')
ax.set_ylabel('Median GMV per tour')
ax.set_yscale('log')
ax.set_ylim(1, max(10, np.nanpercentile(trend['gmv_per_tour'], 99)))
for x, y, n in zip(trend['int_mid'], trend['gmv_per_tour'], trend['suppliers']):
    ax.text(x, y, f" n={n}", fontsize=8)
plt.tight_layout()
plt.show()

# Bookings trend
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(trend['int_mid'], trend['bookings_per_tour'], marker='o', linewidth=2, color=PALETTE['accent'])
ax.set_title('Median bookings per tour vs supplier feature intensity (binned)')
ax.set_xlabel('Avg # core features (bin mid)')
ax.set_ylabel('Median bookings per tour')
ax.set_yscale('log')
ax.set_ylim(1e-2, max(0.1, np.nanpercentile(trend['bookings_per_tour'], 99)))
for x, y, n in zip(trend['int_mid'], trend['bookings_per_tour'], trend['suppliers']):
    ax.text(x, y, f" n={n}", fontsize=8)
plt.tight_layout()
plt.show()

### 5) Feature combinations (bundles): usage and performance

This section answers:
- Which combinations are most common?
- Which combinations correlate with the strongest GMV and bookings per tour?
- Are high-performing bundles actually widespread or niche?

In [0]:

# Bundle performance table
bundle_perf = tour_perf.groupby('bundle').agg(
    tours=('tour_id','count'),
    suppliers=('supplier_id','nunique'),
    gmv_total=('gmv_l12mo','sum'),
    bookings_total=('bookings_l12mo','sum'),
    gmv_per_tour=('gmv_l12mo','median'),
    bookings_per_tour=('bookings_l12mo','median'),
).reset_index()

# Keep stable groups
bundle_perf = bundle_perf[bundle_perf['tours'] >= 200].copy()

# Add shares
bundle_perf['tour_share_pct'] = bundle_perf['tours'] / bundle_perf['tours'].sum() * 100
bundle_perf['gmv_share_pct'] = bundle_perf['gmv_total'] / bundle_perf['gmv_total'].sum() * 100

bundle_perf.sort_values('gmv_per_tour', ascending=False).head(15)

In [0]:

# Visual 1: Most common bundles (by tour share)
d = bundle_perf.sort_values('tour_share_pct', ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(12,7))
ax.barh(d['bundle'], d['tour_share_pct'], color=PALETTE['neutral'], edgecolor='black')
ax.set_title('Most common bundles (share of tours)')
ax.set_xlabel('Share of tours (%)')
for i, (v, n) in enumerate(zip(d['tour_share_pct'], d['tours'])):
    ax.text(v + 0.5, i, f"{v:.1f}%  ({n:,} tours)", va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [0]:

# Visual 2: Best bundles by median GMV per tour

d = bundle_perf.sort_values('gmv_per_tour', ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(12,7))
ax.barh(d['bundle'], d['gmv_per_tour'], color=PALETTE['good'], edgecolor='black')
ax.set_title('Best bundles by median GMV per tour')
ax.set_xlabel('Median GMV per tour (L12M)')
ax.set_xscale('log')
for i, (v, n) in enumerate(zip(d['gmv_per_tour'], d['tours'])):
    ax.text(v, i, f"  {_fmt_eur(v)}  ({n:,} tours)", va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [0]:

# Visual 3: Bubble chart (performance vs prevalence)

# x = tours (prevalence), y = gmv per tour (performance), size = suppliers
b = bundle_perf.copy()

# Limit to top bundles by tours + top by gmv to keep readable
keep = set(b.nlargest(12, 'tours')['bundle']).union(set(b.nlargest(12, 'gmv_per_tour')['bundle']))
b = b[b['bundle'].isin(keep)].copy()

fig, ax = plt.subplots(figsize=(10,6))
ax.scatter(b['tours'], b['gmv_per_tour'], s=np.sqrt(b['suppliers'])*60, alpha=0.7,
           color=PALETTE['primary'], edgecolor='black')

for _, r in b.iterrows():
    ax.annotate(r['bundle'], (r['tours'], r['gmv_per_tour']), xytext=(6,6), textcoords='offset points', fontsize=8)

ax.set_title('Bundles: prevalence vs performance (bubble size = suppliers)')
ax.set_xlabel('Tours in bundle')
ax.set_ylabel('Median GMV per tour')
ax.set_xscale('log')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

### 6) Segment and market pockets: adoption heatmaps + outcome differences

PM questions:
- Which segments lag on adoption?
- In which segments do features correlate with especially strong GMV/bookings differences?

We provide:
- a heatmap of **adoption by segment**
- a heatmap of **within-segment lift** (with vs without)

To prevent clutter, we show the **top segments by tour count**.

In [0]:

# Prepare segment list (top by tours)
if 'segment' not in tour_perf.columns:
    print('No segment column in tour_perf; skip segment cuts.')
else:
    top_segments = tour_perf['segment'].value_counts().head(8).index

    seg_df = tour_perf[tour_perf['segment'].isin(top_segments)].copy()

    # Heatmap 1: adoption by segment
    adopt_seg = seg_df.groupby('segment')[core_features].mean() * 100
    adopt_seg = adopt_seg.rename(columns=label_map)

    mat = adopt_seg.values
    rows = adopt_seg.index.tolist()
    cols = adopt_seg.columns.tolist()

    fig, ax = plt.subplots(figsize=(1.2*len(cols)+4, 0.7*len(rows)+2))
    im = ax.imshow(mat, aspect='auto', vmin=0, vmax=100, cmap='YlGn')

    ax.set_xticks(np.arange(len(cols)))
    ax.set_xticklabels(_maybe_wrap(cols, 12))
    ax.set_yticks(np.arange(len(rows)))
    ax.set_yticklabels(_maybe_wrap(rows, 18))

    for i in range(len(rows)):
        for j in range(len(cols)):
            ax.text(j, i, f"{mat[i,j]:.0f}%", ha='center', va='center', fontsize=9,
                    color='black' if mat[i,j] < 70 else 'white', fontweight='bold')

    ax.set_title('Adoption by segment (top segments)')
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Adoption rate (%)')

    plt.tight_layout()
    plt.show()

In [0]:

# Heatmap 2: within-segment lift (with vs without) on GMV and bookings

if 'segment' in tour_perf.columns:
    top_segments = tour_perf['segment'].value_counts().head(8).index
    seg_df = tour_perf[tour_perf['segment'].isin(top_segments)].copy()

    def within_segment_lift(feature, outcome, min_n=120):
        out = []
        for seg, g in seg_df.groupby('segment'):
            a = g[g[feature]==1][outcome]
            b = g[g[feature]==0][outcome]
            if len(a) >= min_n and len(b) >= min_n and b.mean() > 0:
                lift = (a.mean()/b.mean()-1)*100
                out.append((seg, lift))
            else:
                out.append((seg, np.nan))
        return pd.Series(dict(out))

    # Build matrices
    lift_gmv = pd.DataFrame({label_map.get(f,f): within_segment_lift(f, 'gmv_l12mo') for f in core_features}).loc[top_segments]
    lift_bk  = pd.DataFrame({label_map.get(f,f): within_segment_lift(f, 'bookings_l12mo') for f in core_features}).loc[top_segments]

    for title, m in [('Within-segment GMV lift (%)', lift_gmv), ('Within-segment bookings lift (%)', lift_bk)]:
        mat = m.values
        rows = m.index.tolist()
        cols = m.columns.tolist()

        # Clip for readability
        vmin = np.nanpercentile(mat, 5)
        vmax = np.nanpercentile(mat, 95)
        vmin = max(vmin, -50)
        vmax = min(vmax, 150)

        fig, ax = plt.subplots(figsize=(1.2*len(cols)+4, 0.7*len(rows)+2))
        im = ax.imshow(mat, aspect='auto', vmin=vmin, vmax=vmax, cmap='RdYlGn')

        ax.set_xticks(np.arange(len(cols)))
        ax.set_xticklabels(_maybe_wrap(cols, 12))
        ax.set_yticks(np.arange(len(rows)))
        ax.set_yticklabels(_maybe_wrap(rows, 18))

        for i in range(len(rows)):
            for j in range(len(cols)):
                val = mat[i,j]
                if np.isnan(val):
                    txt = 'NA'
                    color='black'
                else:
                    txt = f"{val:.0f}%"
                    color = 'white' if abs(val) > (0.65*max(abs(vmin), abs(vmax))) else 'black'
                ax.text(j, i, txt, ha='center', va='center', fontsize=9, color=color, fontweight='bold')

        ax.set_title(title)
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label('Lift (%)')

        plt.tight_layout()
        plt.show()

### 7) Operational configuration health (supplier-relevant)

Even if not every operational field is a product surface you own, these charts help a PM understand:
- how suppliers configure operational constraints (cutoff, capacity)
- whether configuration quality correlates with outcomes

We only use fields present in the base table.

In [0]:

# Cutoff distribution (if available)
if 'has_cutoff_configured' in tour.columns and 'cutoff_hours' in tour.columns:
    d = tour[tour['has_cutoff_configured']==1]['cutoff_hours'].dropna()
    if len(d) > 0:
        fig, ax = plt.subplots(figsize=(10,4))
        ax.hist(d.clip(0, d.quantile(0.99)), bins=30, edgecolor='black', color=PALETTE['primary'])
        ax.set_title('Cutoff time distribution (hours) among tours that configured cutoff')
        ax.set_xlabel('Cutoff hours')
        ax.set_ylabel('Tours')
        plt.tight_layout()
        plt.show()

# Vacancy info coverage (if available)
if 'pct_dates_with_vacancy_info' in tour.columns:
    d = tour['pct_dates_with_vacancy_info'].dropna()
    fig, ax = plt.subplots(figsize=(10,4))
    ax.hist(d, bins=np.arange(0, 101, 5), edgecolor='black', color=PALETTE['neutral'])
    ax.set_title('Vacancy information coverage (% dates with vacancy info)')
    ax.set_xlabel('%')
    ax.set_ylabel('Tours')
    plt.tight_layout()
    plt.show()

# Relationship: vacancy coverage vs bookings (hexbin, if both available)
if 'pct_dates_with_vacancy_info' in tour_perf.columns and 'bookings_l12mo' in tour_perf.columns:
    x = tour_perf['pct_dates_with_vacancy_info'].fillna(0)
    y = tour_perf['bookings_l12mo'].fillna(0)

    fig, ax = plt.subplots(figsize=(10,5))
    hb = ax.hexbin(x, np.log1p(y), gridsize=35, cmap='viridis', mincnt=1)
    ax.set_title('Vacancy coverage vs bookings (density)')
    ax.set_xlabel('Vacancy coverage (%)')
    ax.set_ylabel('log(1 + bookings_l12mo)')
    cb = plt.colorbar(hb, ax=ax)
    cb.set_label('Tours (density)')
    plt.tight_layout()
    plt.show()

### 8) PM-ready “What to do next” views (no causal claims)

We create prioritization visuals based on:
- low adoption
- strong raw association (with uncertainty via bootstrap)
- consistency across strata (distribution plots)

This is meant to support **product prioritization** (onboarding, defaults, UX, supplier comms).

In [0]:

# Prioritization scatter: adoption vs raw lift (GMV)

if len(impact) > 0:
    d = impact[impact['outcome']=='gmv_l12mo'].copy()

    # Tour-level adoption
    adopt_map = dict(zip(adopt['feature'], adopt['adoption_rate_pct']))
    d['adoption_rate_pct'] = d['feature'].map(adopt_map)

    fig, ax = plt.subplots(figsize=(10,6))
    ax.scatter(d['adoption_rate_pct'], d['lift_mean_pct'], s=220, alpha=0.85, edgecolor='black', color=PALETTE['accent'])

    for _, r in d.iterrows():
        ax.annotate(r['feature_name'], (r['adoption_rate_pct'], r['lift_mean_pct']), xytext=(6,6), textcoords='offset points', fontsize=9)

    ax.axhline(0, color='black', linewidth=1)
    ax.axvline(np.median(d['adoption_rate_pct']), color=PALETTE['neutral'], linestyle='--', linewidth=1)
    ax.set_title('Prioritization view (descriptive): adoption vs GMV lift')
    ax.set_xlabel('Adoption rate (% tours)')
    ax.set_ylabel('Raw GMV lift (%) with vs without')
    plt.tight_layout()
    plt.show()

In [0]:

# Print a compact PM summary (text output)

lines = []
lines.append('PM SUMMARY (SUPPLIER-SIDE)')
lines.append('')
lines.append(f"Native-pricing tours: {tour['tour_id'].nunique():,}")
lines.append(f"Tours with GMV>0: {tour_perf['tour_id'].nunique():,}")
lines.append(f"Suppliers (GMV>0 tours): {supplier['supplier_id'].nunique():,}")
lines.append('')

# Adoption highlights
lines.append('Adoption landscape (tour-level):')
for _, r in adopt.sort_values('adoption_rate_pct', ascending=False).iterrows():
    lines.append(f"- {r['feature_name']}: {r['adoption_rate_pct']:.1f}% ({r['tours_using']:,} tours)")

lines.append('')

# Lift highlights
if len(impact) > 0:
    g = impact[impact['outcome']=='gmv_l12mo'].sort_values('lift_mean_pct', ascending=False)
    lines.append('Raw GMV association (winsorized mean lift, bootstrap CI):')
    for _, r in g.iterrows():
        lines.append(f"- {r['feature_name']}: {r['lift_mean_pct']:.1f}% (CI {r['ci_low_pct']:.1f} to {r['ci_high_pct']:.1f})")

print('
'.join(lines))

### 9) Notes and recommended next analyses (if you want causality later)

This notebook intentionally avoids regressions and model-based causal claims.

If you later want causal inference, the cleanest upgrade is to add:
- first adoption timestamp per feature per tour

Then you can run:
- pre/post comparisons per tour
- event studies
- difference-in-differences against matched controls

Those require time dimension in the data, which is not assumed here.